# Spatial proteomics case study: MIBI colorectal cancer

This notebook reproduces, with MINA, the **multicellular factor analysis** of the
Hamburg colorectal-cancer (CRC) MIBI tissue-microarray study. Starting from a
single-cell table we:

1. pseudo-bulk **cell-type–stratified features** per field of view (FOV) —
   cell-type abundance, metabolic markers, functional markers and morphology;
2. add a **spatial neighbourhood-interaction view** (MINA's native port of the
   MISTy-style lineage-interaction features) computed from cell centroids;
3. learn **multicellular factors** with MOFA-FLEX;
4. relate factors to **tumour stage** (pT), nodal status (pN) and microsatellite
   instability (MSI), and reconstruct **multicellular coordination networks**.

!!! note "About the data"
    This example expects the study's per-cell table
    `cell_table_with_types_stage.csv` (per-cell marker intensities, morphology,
    centroids, `consensus` cell type, `fov`, and clinical `Stage`/`pN`/`MSI`
    columns). It is **not shipped** with MINA (it is large); set `DATA_DIR`
    below to wherever you keep it.

!!! tip "Requirements"
    `pip install "mina[spatial]"` (brings in `squidpy` for the spatial view and
    `plotnine` for plotting) and a working `mofaflex` install.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
import mudata as md
import matplotlib.pyplot as plt

import mofaflex as mf
import mina

DATA_DIR = Path("path/to/MIBI-Analysis_Hamburg_CRC_TMA_2024/data")
CELL_TABLE = DATA_DIR / "cell_table_with_types_stage.csv"

## 1. Load the single-cell table

We only need the marker, morphology, centroid, cell-type, FOV and clinical
columns.

In [ ]:
metab_markers = ["CA9", "CD98", "CytC", "MCT1", "ASCT2", "LDH", "GS", "GLS",
                 "ATP5A", "CS", "PKM2", "GLUT1", "ARG1", "CPT1A", "Ki67"]
func_markers  = ["PD1", "PDL1", "STING1", "MSH2", "MSH6"]
morpho_feats  = ["eccentricity", "perimeter", "area"]
clinical_cols = ["Stage", "pN group", "MSI gesamt RED"]

needed = set(metab_markers + func_markers + morpho_feats
             + ["centroid-0", "centroid-1", "consensus", "fov"] + clinical_cols)

# index_col is left as default on purpose: combining usecols with index_col=0
# would otherwise consume the first kept marker column as the index.
cell_table = pd.read_csv(CELL_TABLE, usecols=lambda c: c in needed)
cell_table = cell_table[cell_table["consensus"] != "Unclear"]
cell_table.shape

## 2. Define cell-type views and reference FOVs

We group the fine-grained `consensus` labels into eight balanced lineage views.
FOVs are the *samples*. Following the original study we keep only well-staged
FOVs with a sufficient number of epithelial (cancer) cells.

In [ ]:
types_of_interest = dict(
    Other_immune_cell=["APC", "B_cell", "Neutrophil", "Other_immune_cell"],
    Fibroblast=["CAF"],
    Macrophage=["CD163_Macrophage", "CD68_Macrophage"],
    CD4_lymphocyte=["CD4_Tcell", "T_reg_cell"],
    Epithelial_cell=["Cancer_cell"],
    Endothelial_cell=["Endothelial_cell"],
    Monocyte=["Monocyte"],
    Cytotoxic_lymphocyte=["NK_cell", "CD8_Tcell"],
)

epi = cell_table[cell_table["consensus"] == "Cancer_cell"]
dense_fovs = epi["fov"].value_counts()
dense_fovs = dense_fovs[dense_fovs > 20].index

meta = epi[["Stage", "pN group", "MSI gesamt RED", "fov"]]
meta = meta[meta["fov"].isin(dense_fovs)
            & meta["Stage"].isin(["Colon-no.", "pT1", "pT2", "pT3", "pT4"])]
meta_per_fov = meta.groupby("fov").first()
fovs = meta_per_fov.index
len(fovs)

## 3. Build the cell-type–stratified feature views

For each lineage view and FOV we assemble: cell-type **abundance** (proportion),
**metabolic** and **functional** marker medians, and **morphology** (mean and
standard deviation).

In [ ]:
proportions = cell_table.groupby("fov")["consensus"].value_counts().unstack().fillna(0)
proportions = proportions.div(proportions.sum(axis=1), axis=0).reindex(fovs)

features = {}
for view, subtypes in types_of_interest.items():
    present = [s for s in subtypes if s in proportions.columns]
    df = pd.DataFrame(
        proportions.loc[:, present].sum(axis=1).to_numpy().reshape(-1, 1),
        index=fovs, columns=["proportion"],
    )
    sub = cell_table[cell_table["consensus"].isin(subtypes)]
    df = df.join(sub.groupby("fov")[metab_markers].median())
    df = df.join(sub.groupby("fov")[func_markers].median())
    morpho = sub[morpho_feats + ["fov"]]
    df = df.join(morpho.groupby("fov").mean()
                 .join(morpho.groupby("fov").std(), rsuffix="_std"))
    features[view] = df.reindex(fovs).fillna(0)

anndata_dict = {
    view: ad.AnnData(np.asarray(df, dtype=float),
                     obs=pd.DataFrame(index=df.index.astype(str)),
                     var=pd.DataFrame(index=df.columns))
    for view, df in features.items()
}
{v: a.shape for v, a in anndata_dict.items()}

## 4. Add a spatial neighbourhood-interaction view

The original study summarised spatial lineage interactions with MISTy. MINA
ships a native equivalent: `get_nhood_enrichment_feats` builds a
sample × cell-type-pair matrix of Squidpy neighbourhood-enrichment z-scores from
the cell centroids — capturing which lineages co-localise in each FOV.

In [ ]:
spatial_cells = cell_table[cell_table["fov"].isin(fovs)].copy()
sc_adata = ad.AnnData(
    np.zeros((len(spatial_cells), 1)),
    obs=spatial_cells[["fov", "consensus"]].astype(str).reset_index(drop=True),
)
sc_adata.obsm["spatial"] = spatial_cells[["centroid-0", "centroid-1"]].to_numpy()

spatial_view = mina.up.get_nhood_enrichment_feats(
    sc_adata,
    sample_key="fov",
    cluster_key="consensus",
    spatial_key="spatial",
    n_perms=1000,
)

# align the spatial view to the same FOVs / order as the other views
spatial_view = spatial_view[anndata_dict["Epithelial_cell"].obs_names].copy()
spatial_view.X = np.nan_to_num(spatial_view.X)
anndata_dict["Spatial"] = spatial_view

## 5. Normalise, decompose, and assemble the model

We z-score each view (matching the study's per-modality centring and scaling),
prefix features with their view, and learn **10 multicellular factors** with
MOFA-FLEX.

In [ ]:
mina.up.norm_log(anndata_dict, method="zscore")
mina.up.utils.append_view_to_var(anndata_dict)

metadata = meta_per_fov.copy()
metadata.index = metadata.index.astype(str)
metadata["sample_id"] = metadata.index

mdata_model = md.MuData(anndata_dict)
model = mf.terms.MofaFlex(n_factors=10, weight_prior="SpikeSlab", init_factors="pca")
model.fit(mdata_model, seed=0, lr=0.01, early_stopper_patience=1000, subset_var=None,
    save_path=False,
    likelihoods="Normal")

amodel = mina.down.model_to_anndata(
    anndata_dict=anndata_dict, metadata=metadata, model=model
)
amodel

## 6. Variance explained per view and per feature class

In [ ]:
variance_df = mina.down.variance_by_view_info(amodel)
mina.pl.plot_variance_by_view(variance_df)

In [ ]:
feature_type_map = {
    "Abundance": ["proportion"],
    "Metabolism": metab_markers,
    "Function": func_markers,
    "Morphology": morpho_feats + [f"{m}_std" for m in morpho_feats],
}
featureclass_df = mina.down.featureclass_variance_info(amodel, feature_type_map=feature_type_map)
mina.pl.plot_featureclass_variance(featureclass_df)

## 7. Relate factors to clinical stage

We screen every factor against tumour stage (pT), nodal status (pN) and MSI with
Kruskal–Wallis tests, then visualise the leading stage-associated factor.

In [ ]:
scores = mina.down.factor_scores_info(
    amodel, obs_keys=["Stage", "pN group", "MSI gesamt RED"]
)

pT_assoc = mina.down.kruskal_info(scores, group_col="Stage")
pT_assoc.head()

In [ ]:
mina.down.kruskal_info(scores, group_col="MSI gesamt RED").head()

In [ ]:
top_pT_factor = pT_assoc.iloc[0]["factor"]
mina.pl.plot_factor_violin(scores, factor=top_pT_factor, group_col="Stage")

A biplot of the two leading stage-associated factors, with per-group
confidence ellipses, recreates the study's stage-separation figure.

In [ ]:
f_x, f_y = pT_assoc.iloc[0]["factor"], pT_assoc.iloc[1]["factor"]
ellipse_df = mina.down.confidence_ellipses_info(
    scores, x_factor=f_x, y_factor=f_y, group_col="Stage"
)
mina.pl.plot_confidence_ellipses(
    scores, ellipse_df, x_factor=f_x, y_factor=f_y, group_col="Stage"
)

## 8. Top loadings of a stage-associated factor

In [ ]:
loadings = mina.down.variable_loadings_info(amodel)
mina.pl.plot_top_loadings_heatmap(loadings, factor=top_pT_factor, top_n=30)

## 9. Multicellular coordination network

For a factor of interest, `get_multicell_net` selects the top-loading features
per cell-type view, scores their coordinated activity, and fits directed
predictive models between views — yielding a multicellular information network.
Here we use the molecular (non-spatial) views.

In [ ]:
molecular_views = [v for v in anndata_dict if v != "Spatial"]
networks = mina.down.get_multicell_net(
    test_model=amodel,
    sel_factor=top_pT_factor,
    views=molecular_views,
    standardize=True,
    drop_na=True,
    verbose=False,
    percentile=0.6,
)

mina.pl.plot_mcell_network(
    df=networks["negative"],
    weight_col="cor_estimate",
    abs_cutoff=0.3,
    keep_negative=False,
    title=f"Coordination network - {top_pT_factor}",
    show_edge_labels=False,
)

## Summary

Starting from a single-cell MIBI table, MINA pseudo-bulked eight cell-type views
plus a spatial neighbourhood-interaction view, decomposed them into ten
multicellular factors, quantified the variance carried by each lineage and
feature class, linked factors to tumour stage / nodal status / MSI, and
reconstructed multicellular coordination networks — the full multicellular
analysis of the original study, expressed through the MINA API.

The exact factor numbers that associate with each covariate depend on the
decomposition run and seed; screen associations programmatically (as above)
rather than hard-coding factor indices.